<a href="https://colab.research.google.com/github/karsarobert/DeepLearning2026/blob/main/PTE_DL8_FE_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Lab

## Lab 7: Transfer Learning (Feature Extraction)
### November 2, 2026

Source: https://github.com/mrdbourke/tensorflow-deep-learning

# Transfer Learning: Feature Extraction

We've built several convolutional neural networks from scratch, and while they learn, there is still plenty of room for improvement.

To improve our model(s), we could spend time trying different configurations, adding more layers, changing the learning rate, adjusting the number of neurons per layer, and more. However, doing this is very time-consuming.

Fortunately, there is a technique we can use to save time: **transfer learning**. In other words, we take the patterns (also known as weights) that another model has learned from another problem and apply them to our own.

There are two main benefits to using transfer learning:
1. We can leverage an existing neural network architecture proven to work on problems similar to ours.
2. We can leverage a working neural network architecture which has **already learned** patterns on similar data to our own. This often results in achieving great results with less custom data.

Instead of training our own models from scratch on our own datasets, we can take the patterns learned from datasets such as [ImageNet](http://www.image-net.org/) (millions of images of different objects) and use them as the foundation for our own models.


In [ ]:
import os
# Enable Keras 2 compatibility to support tensorflow_hub.KerasLayer (for TF 2.16+)
os.environ['TF_USE_LEGACY_KERAS'] = '1'


## Using a GPU

First, let's check if we are using a GPU. Using a GPU ensures that our model trains much faster than using a CPU alone.


In [ ]:
# Are we using a GPU?
!nvidia-smi

If the cell above doesn't output something that looks like:

```
Fri Sep  4 03:35:21 2020       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 450.66       Driver Version: 418.67       CUDA Version: 10.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla P100-PCIE...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   35C    P0    26W / 250W |      0MiB / 16280MiB |      0%      Default |
|                               |                      |                 ERR! |
+-------------------------------+----------------------+----------------------+
```

Go to Runtime -> Change runtime type -> Hardware accelerator and select "GPU", then re-run the cell above.

## Transfer Learning with Kaggle Models and TensorFlow Hub: Getting great results with 10% of the data

For many problems you may want to use deep learning for, chances are a working model already exists.

The former **TensorFlow Hub (tfhub.dev)** has been fully integrated into **[Kaggle Models](https://www.kaggle.com/models)**. This allows you to import and use a fully trained model using a Kaggle Models URL or the `kagglehub` library.

What if we could get roughly the same (or better) results as our best custom model so far using only 10% of the original data (10x less data)?

**Transfer learning often allows us to achieve great results with less data.**

Let's download a subset of the data we've been using, specifically 10% of the training data from the `10_food_classes` dataset, and use it to train a food image classifier.

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/04-transfer-learning-feature-extraction.png)


## Downloading the Data

In [ ]:
# Get data (10% of labels)
import zipfile

# Download data
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

# Unzip the downloaded file
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip", "r")
zip_ref.extractall()
zip_ref.close()

In [ ]:
# How many images are in each folder?
import os

# Walk through the 10 percent data directory and list number of files
for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

Notice that each of the training directories contains 75 images instead of 750 images. This is key to demonstrating how well transfer learning performs with fewer labeled images.

The test directories still contain the same amount of images (250 per class). This means we will train on less data, but evaluate our models on the same amount of test data.

## Creating Data Loaders (Preparing the Data)

Now that we've downloaded the data, let's use the [`ImageDataGenerator`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) class along with the `flow_from_directory` method to load our images.

In [ ]:
# Setup data inputs
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224, 224)
BATCH_SIZE = 32

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

print("Training images:")
train_data_10_percent = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

print("Testing images:")
# FIXED: Use test_datagen instead of train_datagen, and set shuffle=False for evaluation
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Great! After loading the data, we can see that there are 750 images in the training dataset belonging to 10 classes (75 per class) and 2500 images in the test set belonging to 10 classes (250 per class).

## Building Models with Kaggle Models (formerly TensorFlow Hub)

Now most of the layers in our model will come from **[Kaggle Models](https://www.kaggle.com/models)**.

We are going to compare two architectures:
1. [ResNetV2](https://arxiv.org/abs/1603.05027) - a classic computer vision model architecture from 2016.
2. [EfficientNet](https://arxiv.org/abs/1905.11946) - an efficient computer vision model architecture from 2019.

How to find these models on Kaggle:
1. Go to [kaggle.com/models](https://www.kaggle.com/models).
2. Filter by task (e.g., "Image Classification" or "Image Feature Vector") and framework ("TensorFlow 2" or "Keras").
3. On the selected model page, you can copy the Kaggle Models URL or download it using `kagglehub.model_download()`.

### Three Main Types of Transfer Learning:
1. **"As-is" transfer learning:** taking a pretrained model as it is and applying it to a task without any changes.
2. **Feature extraction transfer learning:** freezing the underlying patterns (weights) learned by the pretrained model (`trainable=False`) and training only the top custom output layer(s) to suit our problem.
3. **Fine-tuning transfer learning:** unfreezing some, many, or all of the layers in the pretrained model and adjusting them to our custom dataset with a low learning rate.

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/04-different-kinds-of-transfer-learning.png)


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import kagglehub
from tensorflow.keras import layers


Now let's get the feature vector URLs for two common computer vision architectures from Kaggle Models: **ResNetV250 (2016)** and **EfficientNetB0 (2019)**.

We are fetching both so we can compare their performance on our dataset.

In [ ]:
# Kaggle Models URLs (legacy tfhub.dev URLs redirect here, but it is best practice to use direct Kaggle links or kagglehub)
resnet_url = "https://www.kaggle.com/models/google/resnet-v2/TensorFlow2/50-feature-vector/2"
efficientnet_url = "https://www.kaggle.com/models/tensorflow/efficientnet/TensorFlow2/b0-feature-vector/1"

# Alternatively, models can be downloaded directly to the local cache using kagglehub:
# resnet_path = kagglehub.model_download("google/resnet-v2/tensorFlow2/50-feature-vector")
# efficientnet_path = kagglehub.model_download("tensorflow/efficientnet/tensorFlow2/b0-feature-vector")


These URLs point to saved pretrained models on Kaggle Models.

When we use them in our model via [`hub.KerasLayer()`](https://www.tensorflow.org/hub/api_docs/python/hub/KerasLayer), the model weights are automatically downloaded.

Since we are comparing two models, let's write a `create_model()` function that takes a Kaggle Models URL (or local path), instantiates a Keras Sequential model with a frozen feature extractor layer and a custom `Dense` output layer, and returns the model.

In [ ]:
def create_model(model_url_or_path, num_classes=10):
  """Takes a Kaggle Models URL or kagglehub path and creates a Keras Sequential model.

  Args:
    model_url_or_path (str): Kaggle Models URL or path returned by kagglehub.
    num_classes (int): Number of output neurons in the output layer (default 10).

  Returns:
    An uncompiled Keras Sequential model with a frozen feature extractor layer.
  """
  feature_extractor_layer = hub.KerasLayer(
      model_url_or_path,
      trainable=False, # freeze the pretrained weights
      name='feature_extraction_layer',
      input_shape=IMAGE_SHAPE + (3,)
  )

  model = tf.keras.Sequential([
      feature_extractor_layer,
      layers.Dense(num_classes, activation='softmax', name='output_layer')
  ])

  return model


Now that we have a function to create a model, let's first build a model using the ResNetV250 architecture as our feature extraction layer.

Once the model is instantiated, we compile it using `categorical_crossentropy` as the loss function, the Adam optimizer, and accuracy as the metric.

In [ ]:
# Create ResNet50V2 model
resnet_model = create_model(resnet_url, num_classes=train_data_10_percent.num_classes)

# Compile
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/04-resnet-feature-extractor.png)
*What our current model looks like: a ResNet50V2 backbone with a custom dense layer on top (10 classes instead of 1000 ImageNet classes).*

Time to fit the model! Let's keep this experiment short and train for 5 epochs.

In [ ]:
# Fit the ResNet model
resnet_history = resnet_model.fit(
    train_data_10_percent,
    epochs=5,
    steps_per_epoch=len(train_data_10_percent),
    validation_data=test_data,
    validation_steps=len(test_data)
)

After just 5 epochs, the ResNetV250 feature extraction model achieves around 90% accuracy on the training set and nearly 80% accuracy on the test set... **using only 10% of the training images!**

Let's inspect our model's training curves using the `plot_loss_curves` function.

In [ ]:

import matplotlib.pyplot as plt

# Plot the validation and training data separately
def plot_loss_curves(history):
  """
  Returns separate loss curves for training and validation metrics.
  """
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  accuracy = history.history['accuracy']
  val_accuracy = history.history['val_accuracy']

  epochs = range(len(history.history['loss']))

  # Plot loss
  plt.plot(epochs, loss, label='training_loss')
  plt.plot(epochs, val_loss, label='val_loss')
  plt.title('Loss')
  plt.xlabel('Epochs')
  plt.legend()

  # Plot accuracy
  plt.figure()
  plt.plot(epochs, accuracy, label='training_accuracy')
  plt.plot(epochs, val_accuracy, label='val_accuracy')
  plt.title('Accuracy')
  plt.xlabel('Epochs')
  plt.legend();

In [ ]:
plot_loss_curves(resnet_history)

Let's also check the summary of our model:

In [ ]:
# Resnet summary
resnet_model.summary()

You can see the power of transfer learning here. The feature extraction layer has 23,564,800 parameters which are pre-learned patterns from ImageNet. Since we set `trainable=False`, these patterns remain frozen during training.

This means that during training, the model only updates the 20,490 parameters in the output layer to suit our dataset.

Now let's do the exact same thing with the **EfficientNetB0** model.

In [ ]:
# Create EfficientNetB0 model
efficientnet_model = create_model(
    model_url_or_path=efficientnet_url,
    num_classes=train_data_10_percent.num_classes
)

# Compile EfficientNet model
efficientnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

# Fit EfficientNet model (for 5 epochs for a fair comparison with ResNet)
efficientnet_history = efficientnet_model.fit(
    train_data_10_percent,
    epochs=5,
    steps_per_epoch=len(train_data_10_percent),
    validation_data=test_data,
    validation_steps=len(test_data)
)


The EfficientNetB0 model performs even better than the ResNetV250 model, achieving over 85% accuracy on the test set—again **with only 10% of the training data**.

Let's check the loss curves.

In [ ]:
plot_loss_curves(efficientnet_history)

From the loss curves of the EfficientNetB0 model, it looks like it could improve even further if we continued training.

Let's inspect the model summary.

In [ ]:
efficientnet_model.summary()

Despite having more than five times fewer parameters (4,049,564 vs. 23,564,800) than the ResNet50V2 extraction layer, the EfficientNetB0 feature extraction layer achieves better performance. Now it is clear where the name "EfficientNet" comes from!

## 📖 Further Reading

* Read through the [TensorFlow Transfer Learning Guide](https://www.tensorflow.org/tutorials/images/transfer_learning) and define the two main types of transfer learning in your own words.
* Browse the available pretrained models on [Kaggle Models](https://www.kaggle.com/models).